# Diabetes Risk Prediction: Phase 1 - Load and Explore Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load the data

In [ ]:
def load_data(path):
    """Load the CSV at `path` into a DataFrame. Raises FileNotFoundError if it does not exist."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at: {path}")
    return pd.read_csv(path)

df = load_data('data/diabetes_data.csv')
df.head()

## 2. Shape, data types and missing values

In [ ]:
print(f"Shape: {df.shape}")
df.info()

In [ ]:
missing_values = df.isnull().sum()
print(f"Total missing values: {missing_values.sum()}")

## 3. Target distribution

In [ ]:
target_distribution = df['Diabetes_binary'].value_counts().sort_index()
target_percentage = df['Diabetes_binary'].value_counts(normalize=True).sort_index() * 100

print(f"No Diabetes (0): {target_distribution[0]:,} ({target_percentage[0]:.2f}%)")
print(f"Diabetes (1): {target_distribution[1]:,} ({target_percentage[1]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
target_distribution.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_xlabel('Diabetes Status')
ax.set_ylabel('Count')
ax.set_title('Distribution of Target Variable')
ax.set_xticklabels(['No Diabetes', 'Diabetes'], rotation=0)
plt.tight_layout()
plt.show()

## 4. Statistical summary

In [1]:
df.describe().T

NameError: name 'df' is not defined

> **Understanding statistical summaries** Statistical summaries provide a quick snapshot of your dataset’s structure and variability.  
>
> - *Central tendency:* Metrics like mean and median reveal typical values but can be distorted by skewed data or outliers.  
> - *Spread:* Standard deviation and quartiles highlight how dispersed the data is, where wide spreads may indicate inconsistent scales or high variability.  
> - *Range and extremes:* Min/max values help spot potential data entry errors or outliers needing review.  
> - *Feature type awareness:*  
>   - Continuous features benefit from scale checks and normalization.  
>   - Ordinal or categorical features should be interpreted by the distribution of categories, not numeric statistics.  
>
> *Why this matters:* Understanding the shape, spread, and nature of your data guides preprocessing decisions like scaling, encoding, and handling outliers for reliable modeling.

## 5. Duplicate rows

In [ ]:
feature_cols = [c for c in df.columns if c != 'Diabetes_binary']

repeat_copies = df.duplicated().sum()                      # rows identical to an earlier row
rows_in_dup_groups = df.duplicated(keep=False).sum()       # includes the first occurrence
conflicting_groups = (df.groupby(feature_cols)['Diabetes_binary'].nunique() > 1).sum()

print(f"Repeat copies (first occurrence kept): {repeat_copies}")
print(f"Rows belonging to a duplicate group: {rows_in_dup_groups}")
print(f"Feature-identical groups with conflicting labels: {conflicting_groups}")
print("\nLabels of rows in duplicate groups:")
print(df[df.duplicated(keep=False)]['Diabetes_binary'].value_counts())

## 6. Correlation with the target

In [ ]:
correlations = df.corr()['Diabetes_binary'].sort_values(ascending=False)
print(correlations)

fig, ax = plt.subplots(figsize=(8, 6))
correlations[1:].plot(kind='barh', ax=ax)  # exclude self-correlation
ax.set_xlabel('Correlation Coefficient')
ax.set_title('Feature Correlations with Diabetes Status')
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Observations

- **Shape and types:** 50,000 rows x 22 columns (21 features + `Diabetes_binary`); every column is `int64`, so no type conversion or categorical encoding is needed.
- **No missing values:** `isnull().sum()` is 0 for every column, so no imputation or row removal is needed.
- **Perfectly balanced target:** 25,000 negative and 25,000 positive (50/50). Accuracy is a fair metric here, but recall still matters most clinically because a false negative is a missed case.
- **Very different feature scales:** `BMI` ranges 12-98, `MentHlth`/`PhysHlth` 0-30, `Age` 1-13, `Income` 1-8, and the binary features 0-1. Features must be scaled (e.g. `StandardScaler`, fit on the training set only) before training.
- **Strongest positive correlations with diabetes:** `GenHlth` (0.41), `HighBP` (0.38), `HighChol` (0.28), `BMI` (0.28), `Age` (0.28), `DiffWalk` (0.27).
- **Protective factors (negative correlations):** `Income` (-0.23), `Education` (-0.17), `PhysActivity` (-0.16), `HvyAlcoholConsump` (-0.10). Income and education are socioeconomic proxies, so fairness across groups should be considered.
- **Weak individual features:** `Sex`, `NoDocbcCost` and `AnyHealthcare` have |r| < 0.05; they may add little on their own.
- **Outliers:** `BMI` reaches 98 (mean about 29.9, std about 7.2), and `MentHlth`/`PhysHlth` are heavily skewed toward 0 with a spike at 30.
- **Duplicate rows:** 938 repeat copies (rows identical to an earlier row across all 22 columns); 1,609 rows sit in duplicate groups (largest group: 12 copies). 222 groups have identical features but conflicting labels. Rows in duplicate groups are 1,370 label 0 vs 239 label 1, so dropping them would unbalance the 50/50 target. Decide how to handle them before splitting so copies do not leak across train and test.
- **Multicollinearity:** `HighBP`, `HighChol` and `BMI` correlate with each other, as do `MentHlth` and `PhysHlth`.

# Phase 2 - Preprocessing

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

TARGET = 'Diabetes_binary'
BATCH_SIZE = 64

## 8. Duplicate handling decision

Measured on this dataset (seed 42, 60/20/20 stratified split, no deduplication):
- 283 of 10,000 test rows (2.8%) have a feature-identical row in the training set, which would inflate test metrics.
- Dropping the 938 repeat copies leaves 49,062 rows (24,186 negative / 24,876 positive, 49.3% / 50.7%), a small shift from 50/50 that stratified splitting preserves in every split.
- 222 feature-identical pairs carry conflicting labels. Dropping them cannot be justified (we do not know which label is right), so they are kept as irreducible label noise.

Decision: drop exact duplicate rows (all 22 columns) **before** splitting.

In [ ]:
def preprocess_data(dataframe, drop_duplicates=True):
    """Drop exact duplicate rows, split into features/target, then into stratified 60/20/20 train/val/test.

    Returns (X_train, X_test, X_val, y_train, y_test, y_val). Raises ValueError if the DataFrame is empty.
    """
    if dataframe.empty:
        raise ValueError("DataFrame is empty")
    if drop_duplicates:
        dataframe = dataframe.drop_duplicates()

    X = dataframe.drop(columns=[TARGET])
    y = dataframe[TARGET]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.4, stratify=y, random_state=RANDOM_SEED)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_SEED)

    return X_train, X_test, X_val, y_train, y_test, y_val

X_train, X_test, X_val, y_train, y_test, y_val = preprocess_data(df)

In [ ]:
total = len(X_train) + len(X_val) + len(X_test)
for name, y_split in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f"{name:5s}: {len(y_split):6d} rows ({len(y_split) / total:.1%}), positive rate {y_split.mean():.2%}")
print(f"full (deduplicated) positive rate: {pd.concat([y_train, y_val, y_test]).mean():.2%}")

In [ ]:
def scale_features(X_train, X_val, X_test):
    """Fit StandardScaler on the training features only; transform train, val and test."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled

X_train_scaled, X_val_scaled, X_test_scaled = scale_features(X_train, X_val, X_test)

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

In [ ]:
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

## 9. Validate DataLoader outputs

In [ ]:
X_batch, y_batch = next(iter(train_loader))
print(f"Feature batch shape: {tuple(X_batch.shape)}  dtype={X_batch.dtype}")
print(f"Label batch shape:   {tuple(y_batch.shape)}  dtype={y_batch.dtype}")
print(f"Unique labels in batch: {y_batch.unique().tolist()}")
print(f"Train feature mean (max abs over features): {X_train_tensor.mean(dim=0).abs().max():.2e}")
print(f"Train feature std (min, max over features): {X_train_tensor.std(dim=0).min():.3f}, {X_train_tensor.std(dim=0).max():.3f}")

# Phase 3 - Model Design

In [ ]:
import torch.nn as nn
from torchinfo import summary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 10. Architecture

`input (num_features) -> Linear(64) -> ReLU -> Linear(32) -> ReLU -> Linear(1) -> Sigmoid`

- Input size is taken from the data (`X_train_tensor.shape[1]`), not hardcoded.
- Two ReLU hidden layers (64, 32): a modest size for about 29k training rows of 21 tabular features. No regularization is used in the baseline (task 004 constraint): the `dropout` constructor argument defaults to 0.0, and dropout is only introduced as a tuning experiment in Phase 6.
- The output is one sigmoid probability per sample, shape `(batch_size, 1)`, so the loss in Phase 4 should be `nn.BCELoss`.

In [ ]:
class DiabetesClassifier(nn.Module):
    def __init__(self, input_size, hidden_sizes=(64, 32), dropout=0.0):
        super().__init__()
        h1, h2 = hidden_sizes
        layers = [nn.Linear(input_size, h1), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers += [nn.Linear(h1, h2), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers += [nn.Linear(h2, 1), nn.Sigmoid()]
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
num_features = X_train_tensor.shape[1]
model = DiabetesClassifier(num_features).to(device)

summary(model, input_size=(BATCH_SIZE, num_features), device=device)

## 11. Forward pass check

In [ ]:
X_batch, y_batch = next(iter(train_loader))
model.eval()
with torch.no_grad():
    output = model(X_batch.to(device))

print(f"Input shape:  {tuple(X_batch.shape)}")
print(f"Output shape: {tuple(output.shape)}")
print(f"Output range: [{output.min():.4f}, {output.max():.4f}]")
print(f"Model device: {next(model.parameters()).device}")

# Phase 4 - Training

## 12. Loss function and optimizer

`nn.BCELoss` (the model already outputs sigmoid probabilities) and `Adam` with `lr=0.001`. The model is re-created after seeding torch so training is reproducible. Labels are `long` tensors, so they are cast to float and reshaped to `(batch, 1)` inside the loop.

In [ ]:
import torch.optim as optim
from typing import List, Tuple

torch.manual_seed(RANDOM_SEED)
model = DiabetesClassifier(num_features).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer.__class__.__name__}, lr={optimizer.param_groups[0]['lr']}")

## 13. Training function

In [ ]:
def train_model(model: nn.Module,
                train_loader: DataLoader,
                val_loader: DataLoader,
                criterion: nn.Module,
                optimizer: optim.Optimizer,
                device: torch.device,
                num_epochs: int = 100,
                print_every: int = 10) -> Tuple[int, List[float], List[float]]:
    """Train a PyTorch model with validation monitoring.

    Returns (num_epochs, train_losses, val_losses); the losses hold one value per epoch.
    """
    train_losses = []
    val_losses = []

    for epoch in range(1, num_epochs + 1):
        # Training phase
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device).float().unsqueeze(1)

                outputs = model(X_batch)
                running_val_loss += criterion(outputs, y_batch).item() * X_batch.size(0)
                correct += ((outputs >= 0.5).float() == y_batch).sum().item()
        val_loss = running_val_loss / len(val_loader.dataset)
        val_accuracy = correct / len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if epoch % print_every == 0 or epoch == 1:
            print(f"Epoch [{epoch}/{num_epochs}] "
                  f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

    print("\nTraining completed!")
    print(f"Final Train Loss: {train_losses[-1]:.4f}")
    print(f"Final Validation Loss: {val_losses[-1]:.4f}")

    return num_epochs, train_losses, val_losses

## 14. Train

In [ ]:
num_epochs, train_losses, val_losses = train_model(
    model, train_loader, val_loader, criterion, optimizer, device,
    num_epochs=100, print_every=10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, num_epochs + 1), train_losses, label='Train loss')
ax.plot(range(1, num_epochs + 1), val_losses, label='Validation loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.set_title('Training and validation loss')
ax.legend()
plt.tight_layout()
plt.show()

# Phase 5 - Evaluation

## 15. Evaluation function

Training loss only shows how well the model fits data it has seen. `evaluate_model` runs the trained model on a held-out loader and returns predictions, labels and accuracy / precision / recall / F1 (computed with `sklearn.metrics`).

In [ ]:
from typing import Dict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model: nn.Module,
                   data_loader: DataLoader,
                   device: torch.device,
                   threshold: float = 0.5) -> Dict:
    """Evaluate a trained model on a dataset and return predictions, labels and metrics.

    Returns a dict with `predictions` (probabilities), `true_labels`, `pred_labels` (binary at `threshold`)
    and `metrics` (accuracy, precision, recall, f1_score).
    """
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            outputs = model(X_batch.to(device))
            all_predictions.append(outputs.squeeze(1).cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())

    predictions = np.concatenate(all_predictions)
    true_labels = np.concatenate(all_labels)
    pred_labels = (predictions >= threshold).astype(int)

    metrics = {
        'accuracy': accuracy_score(true_labels, pred_labels),
        'precision': precision_score(true_labels, pred_labels, zero_division=0),
        'recall': recall_score(true_labels, pred_labels, zero_division=0),
        'f1_score': f1_score(true_labels, pred_labels, zero_division=0),
    }

    return {
        'predictions': predictions,
        'true_labels': true_labels,
        'pred_labels': pred_labels,
        'metrics': metrics,
    }

## 16. Evaluate on the test set

In [ ]:
test_results = evaluate_model(model, test_loader, device)

for name, value in test_results['metrics'].items():
    print(f"{name:>10}: {value:.4f}")

## 17. Confusion matrix

The confusion matrix shows the types of errors the model makes (false positives vs. false negatives), which makes the precision/recall trade-off easier to interpret.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

true_labels = test_results['true_labels']
pred_labels = test_results['pred_labels']
predictions = test_results['predictions']

cm = confusion_matrix(true_labels, pred_labels)
tn, fp, fn, tp = cm.ravel()
print(f"True negatives: {tn} | False positives: {fp} | False negatives: {fn} | True positives: {tp}")

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap='Blues', colorbar=True)
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.show()

## 18. ROC curve and threshold selection

The ROC curve shows the true positive rate against the false positive rate at every decision threshold, and the AUC summarises how well the model separates the two classes. The default threshold (0.5) is marked together with a lower (0.3) and a higher (0.7) threshold.

In [ ]:
fpr, tpr, thresholds = roc_curve(true_labels, predictions)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='#3498db', linewidth=2.5, label=f'Model ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=2, label='Random Classifier (AUC = 0.5)')

# Mark the operating points for the lower, default and higher thresholds
for thr, color in [(0.3, 'green'), (0.5, 'red'), (0.7, 'purple')]:
    idx = np.argmin(np.abs(thresholds - thr))
    plt.scatter(fpr[idx], tpr[idx], color=color, s=150, zorder=5,
                label=f'Threshold {thr}\nTPR={tpr[idx]:.3f}, FPR={fpr[idx]:.3f}')

plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity/Recall)', fontsize=12, fontweight='bold')
plt.title('ROC Curve - Evaluating Threshold Tradeoffs', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nROC Curve Analysis:")
print("=" * 60)
print(f"AUC Score: {roc_auc:.4f}")
print("Interpretation:")
if roc_auc >= 0.90:
    print("  - Excellent discrimination ability")
elif roc_auc >= 0.80:
    print("  - Good discrimination ability")
elif roc_auc >= 0.70:
    print("  - Fair discrimination ability")
else:
    print("  - Poor discrimination ability")

print("\nMetrics at each threshold:")
for thr in (0.3, 0.5, 0.7):
    m = evaluate_model(model, test_loader, device, threshold=thr)['metrics']
    print(f"  threshold={thr}: " + ", ".join(f"{k}={v:.4f}" for k, v in m.items()))